# 05 — Escenarios OPA: base / éxito / fracaso

El núcleo diferencial del proyecto. Proyectamos 90 días bajo tres hipótesis distintas sobre el desenlace de la OPA de BBVA sobre Sabadell.

| Escenario | Hipótesis | Implementación |
|-----------|-----------|----------------|
| **Base** | El periodo de aceptación termina sin movimiento adicional. | `OPA_BBVA_EVENTS` sin modificar. |
| **OPA exitosa** | BBVA absorbe SAB con prima del 30 % sobre cierre. | Evento sintético `opa_exito` con ventana [0,+10]. |
| **OPA fallida** | El Gobierno bloquea o aceptación insuficiente. | Evento sintético `opa_fracaso` con signo opuesto. |

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from prophet import Prophet

from src.data_loader import get_sab, OPA_BBVA_EVENTS

plt.style.use('seaborn-v0_8-whitegrid')
HORIZON = 90

In [ ]:
sab = get_sab(period='5y')
sab['Date'] = pd.to_datetime(sab['Date'])
df = sab[['Date','Close']].rename(columns={'Date':'ds','Close':'y'}).dropna().reset_index(drop=True)
last_date  = df['ds'].max()
last_close = float(df['y'].iloc[-1])
print(f'Última fecha histórico: {last_date.date()}  cierre: {last_close:.3f} EUR')

## 1. Construir tablas de eventos para los 3 escenarios

In [ ]:
# fecha futura simulada para el desenlace OPA -- mediados del horizonte de 90 d
fecha_desenlace = last_date + pd.Timedelta(days=45)
print(f'Fecha desenlace simulado: {fecha_desenlace.date()}')

# Escenario base -- sin nuevos eventos
ev_base = OPA_BBVA_EVENTS.copy()

# Escenario OPA exitosa -- evento positivo grande
ev_exito = pd.concat([
    OPA_BBVA_EVENTS,
    pd.DataFrame({
        'holiday': ['opa_exito'],
        'ds': [fecha_desenlace],
        'lower_window': [0],
        'upper_window': [10],
    })
], ignore_index=True)

# Escenario OPA fallida -- evento negativo
ev_fracaso = pd.concat([
    OPA_BBVA_EVENTS,
    pd.DataFrame({
        'holiday': ['opa_fracaso'],
        'ds': [fecha_desenlace],
        'lower_window': [0],
        'upper_window': [10],
    })
], ignore_index=True)

print('Eventos base:    ', len(ev_base))
print('Eventos éxito:   ', len(ev_exito))
print('Eventos fracaso: ', len(ev_fracaso))

## 2. Función de entrenamiento + proyección por escenario

Entrenamos Prophet con TODO el histórico y proyectamos 90 días. Para forzar la dirección del salto en éxito/fracaso usamos `prior_scale` alta y añadimos un *shock* a posteriori sobre `yhat` proporcional al cierre actual (prima histórica típica de OPA bancarias españolas ≈ ±30 %).

In [ ]:
def forecast_escenario(df, holidays, shock_pct=0.0, shock_date=None, label=''):
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        holidays=holidays,
        holidays_prior_scale=15,   # más peso a los holidays
        interval_width=0.80,
        changepoint_prior_scale=0.10,
    )
    m.add_country_holidays(country_name='ES')
    m.fit(df)
    future = m.make_future_dataframe(periods=HORIZON, freq='D', include_history=False)
    fcst = m.predict(future)

    # aplicar shock multiplicativo a partir de shock_date
    if shock_pct != 0.0 and shock_date is not None:
        mask = fcst['ds'] >= pd.to_datetime(shock_date)
        fcst.loc[mask, ['yhat','yhat_lower','yhat_upper']] *= (1 + shock_pct)

    fcst['escenario'] = label
    return fcst[['ds','yhat','yhat_lower','yhat_upper','escenario']]

In [ ]:
mlflow.set_tracking_uri('file:///' + (ROOT / 'mlruns').as_posix())
mlflow.set_experiment('sab_forecast_escenarios')

with mlflow.start_run(run_name='base'):
    fcst_base = forecast_escenario(df, ev_base, shock_pct=0.0, label='base')
    mlflow.log_param('escenario', 'base')
    mlflow.log_param('shock_pct', 0.0)

with mlflow.start_run(run_name='opa_exito'):
    fcst_exito = forecast_escenario(df, ev_exito, shock_pct=+0.30, shock_date=fecha_desenlace, label='opa_exito')
    mlflow.log_param('escenario', 'opa_exito')
    mlflow.log_param('shock_pct', 0.30)
    mlflow.log_param('fecha_desenlace', str(fecha_desenlace.date()))

with mlflow.start_run(run_name='opa_fracaso'):
    fcst_fracaso = forecast_escenario(df, ev_fracaso, shock_pct=-0.25, shock_date=fecha_desenlace, label='opa_fracaso')
    mlflow.log_param('escenario', 'opa_fracaso')
    mlflow.log_param('shock_pct', -0.25)
    mlflow.log_param('fecha_desenlace', str(fecha_desenlace.date()))

## 3. Tabla resumen — precio esperado a 90 días por escenario

In [ ]:
def resumen(fcst, label):
    fin = fcst.iloc[-1]
    return {
        'Escenario': label,
        'Precio esperado 90d': round(fin['yhat'], 3),
        'IC 80% inf':         round(fin['yhat_lower'], 3),
        'IC 80% sup':         round(fin['yhat_upper'], 3),
        'Variación vs hoy %': round((fin['yhat']/last_close - 1)*100, 2),
    }

tabla = pd.DataFrame([
    resumen(fcst_base,    'Base'),
    resumen(fcst_exito,   'OPA exitosa'),
    resumen(fcst_fracaso, 'OPA fallida'),
]).set_index('Escenario')
tabla

## 4. Visualización comparada

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df['ds'].iloc[-365:], df['y'].iloc[-365:], color='black', label='Histórico (último año)', linewidth=1)

for fcst, color, label in [
    (fcst_base,    'tab:blue',   'Base'),
    (fcst_exito,   'tab:green',  'OPA exitosa (+30%)'),
    (fcst_fracaso, 'tab:red',    'OPA fallida (-25%)'),
]:
    ax.plot(fcst['ds'], fcst['yhat'], color=color, label=label, linewidth=1.5)
    ax.fill_between(fcst['ds'], fcst['yhat_lower'], fcst['yhat_upper'], color=color, alpha=0.10)

ax.axvline(fecha_desenlace, color='gray', linestyle=':', alpha=0.7)
ax.text(fecha_desenlace, ax.get_ylim()[1]*0.98, ' desenlace simulado',
        rotation=90, va='top', fontsize=9, color='gray')
ax.set_title('SAB.MC — Tres escenarios OPA a 90 días')
ax.set_xlabel('Fecha'); ax.set_ylabel('EUR')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 5. Guardar los tres forecasts

In [ ]:
fcst_all = pd.concat([fcst_base, fcst_exito, fcst_fracaso], ignore_index=True)
out = ROOT / 'data' / 'sab_forecast_escenarios_90d.csv'
fcst_all.to_csv(out, index=False)
print(f'Forecast 3 escenarios guardado en: {out}  ({len(fcst_all)} filas)')